# 26. Mínimo de E3 según las relaciones de evolución disponibles

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`10-jss-potencia-criterio-d16.ipynb`](10-jss-potencia-criterio-d16.ipynb) ·
[`src/afg/evaluation/relations.py`](../src/afg/evaluation/relations.py) ·
[`src/afg/experiments/question_bank.py`](../src/afg/experiments/question_bank.py) ·
[D12](../docs/decisions/README.md)

**Estado: completo. Respalda el ADR 0021 (issue #276).** El notebook cuenta cuántas relaciones de
evolución distintas ofrece la referencia de las series de evaluación, deriva de ahí el mínimo de
preguntas E3 que exige `afg gold questions-check --complete` y mide cuánta potencia se pierde por
tener ese número y no 25 (la simulación del notebook 10, con el mismo código).

---

> **Aviso de alcance (ADR 0005).** Las series de este conteo son series de **evaluación**. El
> notebook muestra solo **identificadores de par y conteos**. No muestra ni cita frases de
> decisión de ninguna serie.

## Para qué sirve este análisis

D12 fija que el banco cierra con 25 preguntas por estrato, pero también que, si E3 no alcanza 25
con las series de evaluación cerradas, «se reporta el número real y no se rellena el cupo».
Hasta ahora el cierre exigía 25 en E3 sin condiciones, de modo que un banco que cumple D12 podía
terminar en error. La regla nueva es `mínimo de E3 = min(25, relaciones de evolución disponibles)`,
y las relaciones disponibles se cuentan **desde la referencia**, nunca desde las preguntas
validadas: una pregunta devuelta en la validación sigue bloqueando el cierre hasta que se reescriba.

# 1. Qué se cuenta

Una **relación de evolución disponible** es un par de decisiones (`pair_id`) de una serie de
evaluación cuya relación en la referencia es `refina`, `revierte` o `reemplaza`. Un mismo
`pair_id` cuenta una vez. La serie de evaluación es cada serie de `[oe1].series` que no está en
`[oe1].development_series`. La referencia de cada serie es el archivo que ya resuelve el módulo de
relaciones: `<serie>.candidates.adjudicated.csv` (columna `final_relation`) si existe, o el único
archivo de un anotador `<serie>.candidates.<iniciales>.csv` (columna `relation`).

In [1]:
import pandas as pd

from afg.domain.question import QuestionStratum
from afg.evaluation.relations import (
    EVOLUTION_RELATIONS,
    count_evolution_relations,
    gold_source,
    load_gold_pair_labels,
)
from afg.experiments.question_bank import (
    configured_evolution_relations,
    evaluation_series,
    stratum_minimums,
)
from afg.shared.config import load_experiments_config

BASE_MINIMUM = int(load_experiments_config()["question_bank"]["min_questions"])
SERIES = evaluation_series()
evolution_labels = sorted(r.value for r in EVOLUTION_RELATIONS)
print("Series de evaluación:", len(SERIES), "->", ", ".join(SERIES))
print("Etiquetas que cuentan:", ", ".join(evolution_labels))
print("Mínimo base por estrato (config):", BASE_MINIMUM)

Series de evaluación: 11 -> ES2002, ES2008, ES2014, ES2016, IS1003, IS1006, IS1008, IS1009, TS3003, TS3005, TS3011
Etiquetas que cuentan: reemplaza, refina, revierte
Mínimo base por estrato (config): 25


# 2. Conteo por serie

Para cada serie: qué archivo es su referencia, cuántos pares tienen una etiqueta, cuántos son de
evolución (por etiqueta) y cuáles son (solo identificadores).

In [2]:
rows = []
for series in SERIES:
    source = gold_source(series)
    labels = load_gold_pair_labels(series)
    by_label = {label: len({p for p, r in labels if r == label}) for label in evolution_labels}
    pair_ids = sorted(
        {p for p, r in labels if r in set(evolution_labels)}, key=lambda p: int(p.split(".p")[1])
    )
    rows.append(
        {
            "serie": series,
            "referencia": source.kind,
            "pares etiquetados": len({p for p, _ in labels}),
            **by_label,
            "evolución": count_evolution_relations(labels),
            "pares de evolución": ", ".join(p.split(".")[1] for p in pair_ids) or "-",
        }
    )
TABLE = pd.DataFrame(rows).set_index("serie")
TABLE

,referencia,pares etiquetados,reemplaza,refina,revierte,evolución,pares de evolución
serie,,,,,,,
ES2002,single:gv,9,0,1,1,2,"p006, p007"
ES2008,adjudicated,14,0,5,0,5,"p009, p010, p012, p013, p014"
ES2014,single:gm,14,0,3,0,3,"p004, p010, p011"
ES2016,adjudicated,4,0,2,0,2,"p001, p004"
IS1003,adjudicated,8,1,2,0,3,"p003, p006, p007"
IS1006,single:gv,1,0,0,0,0,-
IS1008,single:gm,8,0,1,0,1,p007
IS1009,single:gv,1,0,0,0,0,-
TS3003,single:gv,6,0,4,0,4,"p001, p003, p005, p006"


In [3]:
AVAILABLE = configured_evolution_relations()
print("Relaciones de evolución disponibles (suma de las series):", AVAILABLE)
assert int(TABLE["evolución"].sum()) == AVAILABLE, (
    "la función configurada y la tabla deben coincidir"
)
print("Series con al menos una:", int((TABLE["evolución"] > 0).sum()), "de", len(SERIES))
print("Series sin ninguna     :", ", ".join(TABLE.index[TABLE["evolución"] == 0]))

Relaciones de evolución disponibles (suma de las series): 24
Series con al menos una: 9 de 11
Series sin ninguna     : IS1006, IS1009


Las **decisiones hijas** de una decisión compuesta no se cuentan aparte. Una relación adjudicada
entre dos decisiones es una sola, y las preguntas que se escriban sobre ella comparten la misma
evidencia anterior y el mismo `superseded_answer`: contarlas como relaciones distintas inflaría el
cupo con preguntas que no aportan información independiente. Por eso el conteo es de `pair_id`
distintos y no de preguntas ni de decisiones.

# 3. El mínimo resultante

In [4]:
MINIMUMS = stratum_minimums(BASE_MINIMUM, AVAILABLE)
pd.Series({s.value: m for s, m in MINIMUMS.items()}, name="mínimo validado").to_frame()

,mínimo validado
E1_point_fact,25
E2_current_state,25
E3_evolution,24
E4_unanswerable,25


El mínimo de E3 pasa de 25 a la cifra de la tabla; los otros tres estratos no cambian. Si la
referencia cambia (por ejemplo, una adjudicación que modifica una etiqueta), el mínimo se
recalcula solo la próxima vez que se corra `questions-check` o `gold status`. Si el conteo
llegara a 25 o más, el mínimo vuelve a ser 25.

# 4. Cuánta potencia cuesta quedarse en el número real

Se reutiliza la simulación del notebook 10 (`afg.experiments.power`): mismas semillas, mismos
remuestreos, 3 corridas por pregunta, exactitud base de C1 = 0,5 y dispersión de dificultad 1
(el escenario de referencia de ese notebook). Se compara el número real de E3 con 25 (el mínimo
anterior) y con 40 (el tope de filas de E3). Cada cifra es el porcentaje de experimentos simulados
que el criterio D16 pareado declara «respaldada»; la columna 0,00 es el falso positivo.

In [5]:
from afg.experiments.power import PowerScenario, power_settings_from_config, simulate_grid

SETTINGS = power_settings_from_config()
RUNS = int(load_experiments_config()["shared_budget"]["runs_per_config"])
SIZES = (MINIMUMS[QuestionStratum.E3_EVOLUTION], BASE_MINIMUM, 40)
DIFFERENCES = (0.0, 0.10, 0.15, 0.20, 0.30)
SCENARIOS = [PowerScenario(n, RUNS, 0.5, d, 1.0) for n in SIZES for d in DIFFERENCES]
RESULTS = simulate_grid(SCENARIOS, SETTINGS, workers=4)
print(len(SCENARIOS), "escenarios,", SETTINGS.n_simulations, "experimentos simulados cada uno")


def table_for(attribute):
    return pd.DataFrame(
        [
            {
                "n": n,
                **{
                    f"{d:+.2f}": round(100 * getattr(r, attribute), 1)
                    for r in RESULTS
                    if r.scenario.n_questions == n
                    for d in [r.scenario.difference]
                },
            }
            for n in SIZES
        ]
    ).set_index("n")


print("Criterio pareado (b), % de experimentos declarados «respaldada»")
table_for("power_paired")

15 escenarios, 1000 experimentos simulados cada uno
Criterio pareado (b), % de experimentos declarados «respaldada»


,+0.00,+0.10,+0.15,+0.20,+0.30
n,,,,,
24,3.8,29.8,53.1,77.0,98.4
25,3.3,31.5,57.5,79.5,98.3
40,2.8,41.1,75.7,93.8,99.8


In [6]:
print("Criterio de no solapamiento (a), % de experimentos declarados «respaldada»")
table_for("power_nonoverlap")

Criterio de no solapamiento (a), % de experimentos declarados «respaldada»


,+0.00,+0.10,+0.15,+0.20,+0.30
n,,,,,
24,0.1,2.7,9.0,24.4,77.0
25,0.0,2.8,10.4,27.8,79.5
40,0.0,4.2,17.4,49.1,96.5


In [7]:
paired = table_for("power_paired")
n_real = SIZES[0]
for diff in ("+0.15", "+0.20"):
    print(
        f"Diferencia {diff}: n={n_real} -> {paired.loc[n_real, diff]} %, "
        f"n={BASE_MINIMUM} -> {paired.loc[BASE_MINIMUM, diff]} %, "
        f"n=40 -> {paired.loc[40, diff]} %. "
        f"Pérdida frente a {BASE_MINIMUM}: "
        f"{round(paired.loc[BASE_MINIMUM, diff] - paired.loc[n_real, diff], 1)} puntos."
    )

Diferencia +0.15: n=24 -> 53.1 %, n=25 -> 57.5 %, n=40 -> 75.7 %. Pérdida frente a 25: 4.4 puntos.
Diferencia +0.20: n=24 -> 77.0 %, n=25 -> 79.5 %, n=40 -> 93.8 %. Pérdida frente a 25: 2.5 puntos.


# 5. Lectura

Con el material disponible, E3 queda en 24 preguntas, una menos que el mínimo anterior. En el
escenario de referencia (dispersión 1, exactitud base de C1 = 0,5) la potencia del criterio
pareado pasa de 57,5 % a 53,1 % con una mejora verdadera de +0,15 (4,4 puntos menos) y de 79,5 % a
77,0 % con +0,20 (2,5 puntos menos). El criterio de no solapamiento baja de 10,4 % a 9,0 % y de
27,8 % a 24,4 %. El falso positivo del pareado es 3,8 % con 24 y 3,3 % con 25, ambos por debajo del
5 %. La pérdida es modesta: de 2,5 a 4,4 puntos, algo por encima del error de la propia simulación
(unos 1,6 puntos), de modo que se lee como una pérdida real pero pequeña. La conclusión del notebook 10 no cambia: el mínimo de E3 es ajustado incluso para el criterio
pareado, y la ganancia grande está en llegar a más preguntas (con 40, 75,7 % y 93,8 %), que el
material de evaluación no permite.

La diferencia de potencia entre 24 y 25 no justifica rellenar el cupo con una pregunta que no
corresponda a una relación genuina: D12 ya lo prohíbe. El faltante frente a 25 se informa como una
limitación de potencia, con estas cifras.

# 6. Límites

- El conteo depende de la referencia de relaciones, que es incompleta por construcción: el
  filtro léxico de la Tarea B dejó fuera relaciones que no se anotaron (ADR 0020, *recall*
  0,407 [0,245; 0,596]). El número disponible es entonces un piso del material real, y se declara
  así, no como el total de relaciones que existen en las reuniones.
- Las cifras de potencia son las de una simulación con datos sintéticos (notebook 10): son
  propiedades del criterio de decisión, no resultados de C1, C2 o C3, y cada una tiene un error de
  Monte Carlo de a lo más unos 1,6 puntos porcentuales. Diferencias de uno o dos puntos no se
  interpretan.
- El mínimo calculado no es una invitación a escribir menos: el tope de 40 filas de E3 sigue
  vigente, y escribir más preguntas genuinas sigue aumentando la potencia.